# 📊 Principal Component Analysis (PCA): From Theory to Practice

---

## 🎯 What This Notebook Covers

This notebook is a **comprehensive, hands-on guide** to PCA (Principal Component Analysis). You will:

1. **Understand the math** behind PCA — covariance, eigenvectors, explained variance
2. **Apply PCA** to a high-dimensional dataset (30 features)
3. **Visualize** the compressed data in 2D and 3D scatter plots
4. **Use PCA** to improve Machine Learning model accuracy
5. **Compare performance** Before vs After PCA across multiple classifiers

---

## 🧠 What is PCA?

**Principal Component Analysis (PCA)** is a dimensionality reduction technique that transforms high-dimensional data into a smaller set of new variables called **principal components**.

These principal components:
- Are **linear combinations** of the original features
- Are **orthogonal** (uncorrelated) to each other
- Are ordered by **how much variance** they explain in the data

### Why Use PCA?
| Problem | PCA Solution |
|---|---|
| Too many features (curse of dimensionality) | Reduces to fewer components |
| Correlated/redundant features | Creates uncorrelated components |
| Slow training due to high dimensions | Faster training on fewer components |
| Hard to visualize high-dim data | Project to 2D/3D for visualization |
| Overfitting risk | Regularization effect via compression |

---

## 📦 Section 1: Install & Import Libraries

We use the following libraries:
- **numpy** — numerical operations
- **pandas** — data manipulation
- **matplotlib / seaborn** — visualization
- **sklearn** — PCA, machine learning models, datasets, metrics

All libraries are pre-installed in Google Colab. Run the cell below to import them.

In [ ]:
# ============================================================
# SECTION 1: LIBRARY IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Sklearn — Dataset
from sklearn.datasets import load_breast_cancer

# Sklearn — Preprocessing
from sklearn.preprocessing import StandardScaler

# Sklearn — Dimensionality Reduction
from sklearn.decomposition import PCA

# Sklearn — Model Selection
from sklearn.model_selection import train_test_split, cross_val_score

# Sklearn — Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Sklearn — Metrics
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

# Utilities
import warnings
import time
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ All libraries imported successfully!')
print(f'NumPy version   : {np.__version__}')
print(f'Pandas version  : {pd.__version__}')

---
## 🗄️ Section 2: Load & Explore the Dataset

We use the **Breast Cancer Wisconsin Dataset** — a classic high-dimensional dataset:
- **569 samples** (patients)
- **30 features** (cell nucleus measurements like radius, texture, perimeter, area...)
- **2 classes**: Malignant (cancerous) and Benign (non-cancerous)

With 30 features, it's a perfect candidate for PCA to reduce dimensionality while preserving meaningful variance.

In [ ]:
# ============================================================
# SECTION 2: LOAD AND EXPLORE DATASET
# ============================================================

# Load the Breast Cancer dataset
cancer = load_breast_cancer()

# Create a DataFrame for easy exploration
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

# Basic dataset information
print('=' * 55)
print('         BREAST CANCER DATASET — OVERVIEW')
print('=' * 55)
print(f'  Total Samples   : {df.shape[0]}')
print(f'  Total Features  : {len(cancer.feature_names)}')
print(f'  Target Classes  : {list(cancer.target_names)}')
print(f'  Class Balance   : {dict(df["diagnosis"].value_counts())}')
print('=' * 55)

print('\n📋 First 5 rows of the dataset:')
df.head()

In [ ]:
# ============================================================
# SECTION 2B: DESCRIPTIVE STATISTICS & CLASS DISTRIBUTION
# ============================================================

print('📊 Statistical Summary of First 10 Features:')
print(df[cancer.feature_names[:10]].describe().round(3))

print('\n\n🏷️  Class Distribution:')
print(df['diagnosis'].value_counts())

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = df['diagnosis'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=1.2)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', fontweight='bold', fontsize=13)

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            colors=['#e74c3c', '#2ecc71'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Breast Cancer Dataset — Class Balance', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚙️ Section 3: Preprocessing — Feature Scaling

### Why Scale Before PCA?

PCA is **sensitive to the scale of features**. A feature measured in thousands (e.g., area) will dominate over one measured in decimals (e.g., texture). 

We apply **StandardScaler** which transforms each feature to:
- **Mean = 0**
- **Standard Deviation = 1**

Formula: `z = (x - μ) / σ`

This ensures all 30 features contribute equally to PCA.

In [ ]:
# ============================================================
# SECTION 3: DATA PREPROCESSING — STANDARDIZATION
# ============================================================

# Extract features and labels
X = cancer.data          # Shape: (569, 30)
y = cancer.target        # Shape: (569,)

print(f'Feature matrix X shape : {X.shape}')
print(f'Target vector y shape  : {y.shape}')

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verify scaling
print('\n✅ After Standardization:')
print(f'  Mean (should be ~0)  : {X_scaled.mean(axis=0)[:5].round(10)}')
print(f'  Std  (should be ~1)  : {X_scaled.std(axis=0)[:5].round(10)}')

# Visualize before vs after scaling for one feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X[:, 0], bins=30, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Before Scaling\n(mean radius — raw values)', fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(X_scaled[:, 0], bins=30, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('After Standardization\n(mean radius — z-score)', fontweight='bold')
axes[1].set_xlabel('Z-Score (σ units)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Feature Scaling Effect', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 Section 4: Applying PCA — Explained Variance Analysis

### How PCA Works (Step by Step)

1. **Compute Covariance Matrix** — measures how features vary together
2. **Compute Eigenvectors & Eigenvalues** — eigenvectors = directions of max variance; eigenvalues = amount of variance
3. **Sort by Eigenvalue** — largest eigenvalue = most important direction
4. **Project Data** — transform data onto the top k eigenvectors

The **explained variance ratio** tells us what % of total information each component captures.

> **Rule of Thumb**: Keep enough components to explain ≥ 95% of the variance.

In [ ]:
# ============================================================
# SECTION 4: PCA — EXPLAINED VARIANCE ANALYSIS
# ============================================================

# Fit PCA keeping ALL components to analyze variance
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

# Explained variance per component
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# How many components for 95% variance?
n_95 = np.argmax(cumulative_var >= 0.95) + 1
n_99 = np.argmax(cumulative_var >= 0.99) + 1

print('=' * 50)
print('     PCA EXPLAINED VARIANCE SUMMARY')
print('=' * 50)
print(f'  Original features       : {X_scaled.shape[1]}')
print(f'  Components for 90% var  : {np.argmax(cumulative_var >= 0.90) + 1}')
print(f'  Components for 95% var  : {n_95}')
print(f'  Components for 99% var  : {n_99}')
print(f'  PC1 explains            : {explained_var[0]*100:.2f}%')
print(f'  PC2 explains            : {explained_var[1]*100:.2f}%')
print(f'  PC1+PC2 combined        : {cumulative_var[1]*100:.2f}%')
print('=' * 50)

print('\n📋 Variance per component (first 10):')
variance_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(10)],
    'Explained Variance (%)': (explained_var[:10] * 100).round(2),
    'Cumulative Variance (%)': (cumulative_var[:10] * 100).round(2)
})
print(variance_df.to_string(index=False))

In [ ]:
# ============================================================
# SECTION 4B: SCREE PLOT & CUMULATIVE VARIANCE PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

components = range(1, len(explained_var) + 1)

# --- LEFT: Scree Plot ---
axes[0].bar(list(components)[:15], explained_var[:15] * 100,
            color='steelblue', edgecolor='navy', alpha=0.85)
axes[0].plot(list(components)[:15], explained_var[:15] * 100,
             'ro-', linewidth=2, markersize=7, label='Individual variance')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance (%)', fontsize=12)
axes[0].set_title('Scree Plot\n(Individual Explained Variance per PC)', fontsize=13, fontweight='bold')
axes[0].set_xticks(list(components)[:15])
axes[0].legend()

# --- RIGHT: Cumulative Variance ---
axes[1].plot(list(components), cumulative_var * 100,
             'b-o', linewidth=2.5, markersize=5, label='Cumulative variance')
axes[1].axhline(y=95, color='red', linestyle='--', linewidth=2, label='95% threshold')
axes[1].axhline(y=99, color='green', linestyle='--', linewidth=2, label='99% threshold')
axes[1].axvline(x=n_95, color='red', linestyle=':', linewidth=1.5,
                label=f'{n_95} components = 95%')
axes[1].axvline(x=n_99, color='green', linestyle=':', linewidth=1.5,
                label=f'{n_99} components = 99%')
axes[1].fill_between(list(components)[:n_95], cumulative_var[:n_95] * 100,
                     alpha=0.15, color='red', label='Region for 95% coverage')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=12)
axes[1].set_title('Cumulative Explained Variance\n(How many components do we need?)',
                  fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].set_ylim([40, 102])
axes[1].grid(True, alpha=0.4)

plt.suptitle('PCA Variance Analysis — Breast Cancer Dataset (30 Features)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n💡 Insight: Just {n_95} components (out of 30) capture 95% of all information!')

---
## 📈 Section 5: Dimensionality Reduction & 2D Scatter Plot

We now **project the 30-dimensional data onto just 2 principal components** to visualize it in 2D. Although 2 components only explain ~63% of the variance, this is enough to visually separate the two classes (Malignant vs Benign).

The scatter plot reveals whether PCA found a meaningful structure in the data.

In [ ]:
# ============================================================
# SECTION 5: PROJECT TO 2D AND VISUALIZE
# ============================================================

# Apply PCA — reduce to 2 components for visualization
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f'Original shape  : {X_scaled.shape}  →  {X_pca_2d.shape} after PCA')
print(f'PC1 explains    : {pca_2d.explained_variance_ratio_[0]*100:.2f}%')
print(f'PC2 explains    : {pca_2d.explained_variance_ratio_[1]*100:.2f}%')
print(f'Total retained  : {pca_2d.explained_variance_ratio_.sum()*100:.2f}%')

# Create the 2D PCA DataFrame
pca_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_df['Diagnosis'] = ['Malignant' if t == 0 else 'Benign' for t in y]

print('\n📋 PCA-transformed data (first 5 rows):')
print(pca_df.head())

In [ ]:
# ============================================================
# SECTION 5B: 2D PCA SCATTER PLOT (MAIN VISUALIZATION)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors = {'Malignant': '#e74c3c', 'Benign': '#2ecc71'}
markers = {'Malignant': 'X', 'Benign': 'o'}

# --- LEFT: Basic Scatter Plot ---
for diagnosis, color in colors.items():
    mask = pca_df['Diagnosis'] == diagnosis
    axes[0].scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        c=color, marker=markers[diagnosis],
        alpha=0.7, edgecolors='white', linewidth=0.5,
        s=60, label=diagnosis
    )

axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
axes[0].set_title('PCA — 2D Projection\n(30 features → 2 components)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=12, markerscale=1.5)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.4)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.4)

# --- RIGHT: KDE Density Plot ---
for diagnosis, color in colors.items():
    mask = pca_df['Diagnosis'] == diagnosis
    axes[1].scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        c=color, alpha=0.4, s=40, label=diagnosis
    )
    # Add contour
    sns.kdeplot(
        x=pca_df.loc[mask, 'PC1'],
        y=pca_df.loc[mask, 'PC2'],
        ax=axes[1], color=color,
        levels=4, linewidths=2
    )

axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
axes[1].set_title('PCA — 2D Projection with Density Contours\n(Class Separation Visualization)',
                  fontsize=13, fontweight='bold')
axes[1].legend(fontsize=12)

plt.suptitle('PCA 2D Scatter Plot — Breast Cancer Dataset',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n💡 Observation: Even with only 2 components, there is clear separation')
print('   between Malignant (red) and Benign (green) clusters!')

---
## 🌐 Section 6: 3D PCA Scatter Plot

With 3 components we retain even more variance. The 3D plot gives a richer view of the cluster structure in the PCA-reduced space.

In [ ]:
# ============================================================
# SECTION 6: 3D PCA SCATTER PLOT
# ============================================================

# Reduce to 3 components
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3d = pca_3d.fit_transform(X_scaled)

total_3d = pca_3d.explained_variance_ratio_.sum()
print(f'3 components explain: {total_3d*100:.2f}% of total variance')

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

color_map = {0: '#e74c3c', 1: '#2ecc71'}
label_map = {0: 'Malignant', 1: 'Benign'}

for label in [0, 1]:
    mask = y == label
    ax.scatter(
        X_pca_3d[mask, 0],
        X_pca_3d[mask, 1],
        X_pca_3d[mask, 2],
        c=color_map[label],
        label=label_map[label],
        alpha=0.7, s=50, edgecolors='white', linewidth=0.3
    )

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)', fontsize=11)
ax.set_title(f'3D PCA Scatter Plot\n({total_3d*100:.1f}% variance retained)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12, markerscale=1.5)
ax.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

---
## 🔍 Section 7: Understanding PCA Loadings — Feature Contributions

The **PCA loadings** (eigenvectors) tell us which original features contribute most to each principal component. A high absolute loading value means that feature has a strong influence on that component.

This is important for **interpretability** — understanding WHAT the PCA is capturing from the original features.

In [ ]:
# ============================================================
# SECTION 7: PCA LOADINGS — FEATURE CONTRIBUTION ANALYSIS
# ============================================================

# Extract loadings for the 2D PCA
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=['PC1', 'PC2'],
    index=cancer.feature_names
)

# Sort by absolute value of PC1
loadings_sorted = loadings.reindex(loadings['PC1'].abs().sort_values(ascending=False).index)

print('Top 10 features contributing to PC1 (most influential):')
print(loadings_sorted['PC1'].head(10).round(4))

# Heatmap of loadings
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Heatmap
sns.heatmap(
    loadings,
    ax=axes[0],
    cmap='RdBu_r', center=0,
    annot=True, fmt='.2f',
    linewidths=0.5, linecolor='gray',
    cbar_kws={'label': 'Loading value'}
)
axes[0].set_title('PCA Loadings Heatmap\n(Feature contributions to PC1 & PC2)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Principal Components', fontsize=12)
axes[0].set_ylabel('Original Features', fontsize=12)

# Bar chart — top features for PC1
top_features = loadings_sorted['PC1'].head(15)
colors_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in top_features.values]
axes[1].barh(range(len(top_features)), top_features.values,
             color=colors_bar, edgecolor='black', alpha=0.85)
axes[1].set_yticks(range(len(top_features)))
axes[1].set_yticklabels(top_features.index, fontsize=10)
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_xlabel('Loading Value', fontsize=12)
axes[1].set_title('Top 15 Feature Loadings on PC1\n(How each original feature influences PC1)',
                  fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n💡 Positive loading = feature pushes sample to right on PC1 axis')
print('   Negative loading = feature pushes sample to left on PC1 axis')

---
## 🤖 Section 8: Machine Learning — Before PCA (Baseline Models)

We train 4 classifiers on the **original 30 features** to establish baseline performance:

| Model | How it works |
|---|---|
| **Logistic Regression** | Linear decision boundary using sigmoid function |
| **Support Vector Machine** | Finds optimal hyperplane with maximum margin |
| **Random Forest** | Ensemble of decision trees, votes by majority |
| **K-Nearest Neighbors** | Classifies based on k closest training samples |

We measure: **Accuracy, ROC-AUC, Training Time**.

In [ ]:
# ============================================================
# SECTION 8: TRAIN/TEST SPLIT
# ============================================================

# Split scaled data (80% train, 20% test) — stratified to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train/Test Split Summary:')
print(f'  Training samples : {X_train.shape[0]} ({X_train.shape[1]} features each)')
print(f'  Testing samples  : {X_test.shape[0]} ({X_test.shape[1]} features each)')
print(f'  Train class dist : Malignant={sum(y_train==0)}, Benign={sum(y_train==1)}')
print(f'  Test class dist  : Malignant={sum(y_test==0)}, Benign={sum(y_test==1)}')

In [ ]:
# ============================================================
# SECTION 8B: TRAIN MODELS ON ORIGINAL 30 FEATURES (BASELINE)
# ============================================================

# Define classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=RANDOM_STATE),
    'Support Vector Machine': SVC(probability=True, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

# Store baseline results
baseline_results = {}

print('🔄 Training models on ORIGINAL 30 features...')
print('=' * 60)

for name, clf in classifiers.items():
    start_time = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start_time

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cv_scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')

    baseline_results[name] = {
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'CV Mean Accuracy': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Train Time (s)': train_time
    }

    print(f'{name}')
    print(f'  Accuracy   : {accuracy*100:.2f}%')
    print(f'  ROC-AUC    : {roc_auc:.4f}')
    print(f'  CV (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
    print(f'  Train time : {train_time:.4f} seconds')
    print('-' * 60)

print('\n✅ Baseline training complete!')

---
## 🚀 Section 9: Machine Learning — After PCA (Reduced Features)

Now we apply PCA to reduce 30 features to **10 components** (which cover ~95% of the variance) and retrain the same models. We then compare:

- Does accuracy improve or stay the same?
- Does training speed up?
- Is there a tradeoff between dimensionality and performance?

In [ ]:
# ============================================================
# SECTION 9: APPLY PCA THEN RETRAIN MODELS
# ============================================================

# IMPORTANT: Fit PCA ONLY on training data, then transform both train and test
# This prevents data leakage from test set into PCA components
n_components_pca = n_95  # Use number needed for 95% variance

pca_ml = PCA(n_components=n_components_pca, random_state=RANDOM_STATE)
X_train_pca = pca_ml.fit_transform(X_train)  # Fit on train only
X_test_pca = pca_ml.transform(X_test)        # Transform test (no fitting)

print(f'PCA applied with {n_components_pca} components (95% variance threshold)')
print(f'  Train shape before PCA : {X_train.shape}')
print(f'  Train shape after PCA  : {X_train_pca.shape}')
print(f'  Test  shape before PCA : {X_test.shape}')
print(f'  Test  shape after PCA  : {X_test_pca.shape}')
print(f'  Variance retained      : {pca_ml.explained_variance_ratio_.sum()*100:.2f}%')

print('\n⚠️  NOTE: PCA was fit on training data ONLY to prevent data leakage!')

# Store PCA results
pca_results = {}

print('\n🔄 Training models on PCA-reduced features...')
print('=' * 60)

# Use fresh instances of classifiers
classifiers_pca = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=RANDOM_STATE),
    'Support Vector Machine': SVC(probability=True, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

X_scaled_pca = pca_ml.transform(X_scaled)  # For cross-validation

for name, clf in classifiers_pca.items():
    start_time = time.time()
    clf.fit(X_train_pca, y_train)
    train_time = time.time() - start_time

    y_pred = clf.predict(X_test_pca)
    y_proba = clf.predict_proba(X_test_pca)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cv_scores = cross_val_score(clf, X_scaled_pca, y, cv=5, scoring='accuracy')

    pca_results[name] = {
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'CV Mean Accuracy': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Train Time (s)': train_time
    }

    delta = (accuracy - baseline_results[name]['Accuracy']) * 100
    arrow = '⬆️' if delta > 0 else ('⬇️' if delta < 0 else '➡️')

    print(f'{name}')
    print(f'  Accuracy   : {accuracy*100:.2f}%  {arrow} ({delta:+.2f}% vs baseline)')
    print(f'  ROC-AUC    : {roc_auc:.4f}')
    print(f'  CV (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
    print(f'  Train time : {train_time:.4f} seconds')
    print('-' * 60)

print('\n✅ PCA-enhanced training complete!')

---
## 📊 Section 10: Comprehensive Before vs After Comparison

This section brings everything together in a **side-by-side comparison** of all models, showing:
- Accuracy before and after PCA
- ROC-AUC before and after PCA
- Training time reduction
- Dimensionality reduction ratio

In [ ]:
# ============================================================
# SECTION 10: BEFORE vs AFTER PCA — COMPARISON TABLE
# ============================================================

print('\n' + '=' * 80)
print('          PERFORMANCE COMPARISON: BEFORE vs AFTER PCA')
print(f'          (Original: 30 features → PCA: {n_components_pca} components)')
print('=' * 80)

comparison_data = []

for name in classifiers.keys():
    b = baseline_results[name]
    p = pca_results[name]

    acc_delta = (p['Accuracy'] - b['Accuracy']) * 100
    auc_delta = p['ROC-AUC'] - b['ROC-AUC']
    time_delta = ((b['Train Time (s)'] - p['Train Time (s)']) / b['Train Time (s)']) * 100

    comparison_data.append({
        'Model': name,
        'Acc Before (%)': f"{b['Accuracy']*100:.2f}",
        'Acc After (%)': f"{p['Accuracy']*100:.2f}",
        'Acc Δ': f"{acc_delta:+.2f}",
        'AUC Before': f"{b['ROC-AUC']:.4f}",
        'AUC After': f"{p['ROC-AUC']:.4f}",
        'AUC Δ': f"{auc_delta:+.4f}",
        'Time Saved (%)': f"{time_delta:+.1f}"
    })

comp_df = pd.DataFrame(comparison_data)
print(comp_df.to_string(index=False))
print('=' * 80)
print(f'\n📉 Dimensionality Reduction : {X_scaled.shape[1]} → {n_components_pca} features')
print(f'   That is a {(1 - n_components_pca/X_scaled.shape[1])*100:.1f}% reduction in dimensions!')

In [ ]:
# ============================================================
# SECTION 10B: COMPARISON BAR CHARTS
# ============================================================

model_names = list(classifiers.keys())
short_names = ['LR', 'SVM', 'RF', 'KNN']  # Abbreviated for plot

acc_before = [baseline_results[m]['Accuracy'] * 100 for m in model_names]
acc_after = [pca_results[m]['Accuracy'] * 100 for m in model_names]

auc_before = [baseline_results[m]['ROC-AUC'] for m in model_names]
auc_after = [pca_results[m]['ROC-AUC'] for m in model_names]

time_before = [baseline_results[m]['Train Time (s)'] for m in model_names]
time_after = [pca_results[m]['Train Time (s)'] for m in model_names]

x = np.arange(len(model_names))
bar_w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# --- Plot 1: Accuracy ---
bars1 = axes[0].bar(x - bar_w/2, acc_before, bar_w, label='Before PCA (30 features)',
                    color='#3498db', edgecolor='navy', alpha=0.85)
bars2 = axes[0].bar(x + bar_w/2, acc_after, bar_w, label=f'After PCA ({n_components_pca} components)',
                    color='#e74c3c', edgecolor='darkred', alpha=0.85)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[0].set_xticks(x)
axes[0].set_xticklabels(short_names, fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy\nBefore vs After PCA', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim([85, 102])

# --- Plot 2: ROC-AUC ---
axes[1].bar(x - bar_w/2, auc_before, bar_w, label='Before PCA',
            color='#2ecc71', edgecolor='darkgreen', alpha=0.85)
axes[1].bar(x + bar_w/2, auc_after, bar_w, label=f'After PCA',
            color='#f39c12', edgecolor='darkorange', alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(short_names, fontsize=12)
axes[1].set_ylabel('ROC-AUC Score', fontsize=12)
axes[1].set_title('ROC-AUC Score\nBefore vs After PCA', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim([0.92, 1.01])

# --- Plot 3: Training Time ---
axes[2].bar(x - bar_w/2, time_before, bar_w, label='Before PCA',
            color='#9b59b6', edgecolor='indigo', alpha=0.85)
axes[2].bar(x + bar_w/2, time_after, bar_w, label='After PCA',
            color='#1abc9c', edgecolor='teal', alpha=0.85)

axes[2].set_xticks(x)
axes[2].set_xticklabels(short_names, fontsize=12)
axes[2].set_ylabel('Training Time (seconds)', fontsize=12)
axes[2].set_title('Training Time\nBefore vs After PCA', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)

plt.suptitle(f'Model Performance Comparison — 30 Features vs {n_components_pca} PCA Components',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 10C: CROSS-VALIDATION COMPARISON (5-FOLD)
# ============================================================

cv_before = [baseline_results[m]['CV Mean Accuracy'] * 100 for m in model_names]
cv_after = [pca_results[m]['CV Mean Accuracy'] * 100 for m in model_names]
cv_std_before = [baseline_results[m]['CV Std'] * 100 for m in model_names]
cv_std_after = [pca_results[m]['CV Std'] * 100 for m in model_names]

fig, ax = plt.subplots(figsize=(12, 6))

ax.errorbar(short_names, cv_before, yerr=cv_std_before,
            fmt='b-o', linewidth=2.5, markersize=10, capsize=8,
            label='Before PCA (30 features)', color='#3498db')
ax.errorbar(short_names, cv_after, yerr=cv_std_after,
            fmt='r-s', linewidth=2.5, markersize=10, capsize=8,
            label=f'After PCA ({n_components_pca} components)', color='#e74c3c')

# Fill between
ax.fill_between(short_names,
                [c - s for c, s in zip(cv_before, cv_std_before)],
                [c + s for c, s in zip(cv_before, cv_std_before)],
                alpha=0.15, color='blue')
ax.fill_between(short_names,
                [c - s for c, s in zip(cv_after, cv_std_after)],
                [c + s for c, s in zip(cv_after, cv_std_after)],
                alpha=0.15, color='red')

ax.set_ylabel('Cross-Validation Accuracy (%)', fontsize=12)
ax.set_title('5-Fold Cross-Validation Accuracy\nBefore vs After PCA (with ±1σ error bands)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim([88, 102])
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

---
## 🧪 Section 11: Optimal Number of PCA Components — Grid Search

Instead of just testing one PCA dimension, we now run a **grid search** over different numbers of components and see how Logistic Regression accuracy changes. This shows the tradeoff between compression and information loss.

In [ ]:
# ============================================================
# SECTION 11: OPTIMAL PCA COMPONENTS — GRID SEARCH
# ============================================================

component_range = range(1, 31)  # Try 1 to 30 components
lr_accuracies = []
svm_accuracies = []

for n in component_range:
    pca_temp = PCA(n_components=n, random_state=RANDOM_STATE)
    X_tr = pca_temp.fit_transform(X_train)
    X_te = pca_temp.transform(X_test)

    # Logistic Regression
    lr = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
    lr.fit(X_tr, y_train)
    lr_accuracies.append(accuracy_score(y_test, lr.predict(X_te)) * 100)

    # SVM
    svm = SVC(probability=False, random_state=RANDOM_STATE)
    svm.fit(X_tr, y_train)
    svm_accuracies.append(accuracy_score(y_test, svm.predict(X_te)) * 100)

# Baseline accuracies
lr_baseline = baseline_results['Logistic Regression']['Accuracy'] * 100
svm_baseline = baseline_results['Support Vector Machine']['Accuracy'] * 100

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(list(component_range), lr_accuracies, 'b-o', linewidth=2,
        markersize=5, label='Logistic Regression')
ax.plot(list(component_range), svm_accuracies, 'r-s', linewidth=2,
        markersize=5, label='SVM')

ax.axhline(y=lr_baseline, color='blue', linestyle='--', alpha=0.5,
           label=f'LR Baseline (30 feat): {lr_baseline:.1f}%')
ax.axhline(y=svm_baseline, color='red', linestyle='--', alpha=0.5,
           label=f'SVM Baseline (30 feat): {svm_baseline:.1f}%')

ax.axvline(x=n_95, color='green', linestyle=':', linewidth=2,
           label=f'{n_95} components = 95% variance')

ax.set_xlabel('Number of PCA Components', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Number of PCA Components\n(Optimal compression point)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xticks(list(component_range))
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

best_n_lr = list(component_range)[np.argmax(lr_accuracies)]
print(f'\n💡 LR best accuracy at {best_n_lr} components : {max(lr_accuracies):.2f}%')
print(f'   SVM best accuracy at {list(component_range)[np.argmax(svm_accuracies)]} components : {max(svm_accuracies):.2f}%')

---
## 📋 Section 12: Confusion Matrix & Classification Report (After PCA)

The **confusion matrix** shows the breakdown of correct and incorrect predictions:
- **True Positive (TP)**: Malignant correctly predicted
- **True Negative (TN)**: Benign correctly predicted
- **False Positive (FP)**: Benign predicted as Malignant (Type I error)
- **False Negative (FN)**: Malignant predicted as Benign (Type II error — most dangerous in medical context!)

In [ ]:
# ============================================================
# SECTION 12: CONFUSION MATRICES (AFTER PCA) FOR ALL MODELS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for idx, (name, clf) in enumerate(classifiers_pca.items()):
    y_pred = clf.predict(X_test_pca)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm, annot=True, fmt='d', ax=axes[idx],
        cmap='Blues', linewidths=1,
        xticklabels=['Malignant', 'Benign'],
        yticklabels=['Malignant', 'Benign'],
        annot_kws={'size': 16, 'fontweight': 'bold'}
    )
    acc = accuracy_score(y_test, y_pred)
    axes[idx].set_title(f'{name}\nAccuracy: {acc*100:.2f}%', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    axes[idx].set_ylabel('True Label', fontsize=11)

plt.suptitle(f'Confusion Matrices — After PCA ({n_components_pca} Components)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Detailed classification report for best model
best_model_name = max(pca_results, key=lambda m: pca_results[m]['Accuracy'])
best_clf = classifiers_pca[best_model_name]
y_pred_best = best_clf.predict(X_test_pca)

print(f'\n📋 Detailed Classification Report — {best_model_name} (Best Model After PCA)')
print('=' * 60)
print(classification_report(y_test, y_pred_best, target_names=['Malignant', 'Benign']))

---
## 📝 Section 13: Final Summary & Conclusions

A complete summary of everything we did, and what the results mean.

In [ ]:
# ============================================================
# SECTION 13: FINAL SUMMARY DASHBOARD
# ============================================================

print('\n' + '█' * 65)
print('              FINAL RESULTS SUMMARY')
print('█' * 65)
print(f"""
  Dataset           : Breast Cancer Wisconsin
  Original Features : 30
  PCA Components    : {n_components_pca} (captures {pca_ml.explained_variance_ratio_.sum()*100:.1f}% variance)
  Dimension Reduction: {(1 - n_components_pca/30)*100:.1f}% fewer features
""")

print('  {:25s} {:>12s} {:>12s} {:>10s}'.format(
    'Model', 'Before PCA', 'After PCA', 'Change'))
print('  ' + '-' * 62)

for name in model_names:
    b_acc = baseline_results[name]['Accuracy'] * 100
    a_acc = pca_results[name]['Accuracy'] * 100
    delta = a_acc - b_acc
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '→')
    print(f'  {name:25s} {b_acc:>10.2f}%  {a_acc:>10.2f}%  {arrow}{abs(delta):>6.2f}%')

print('█' * 65)

print("""
  KEY TAKEAWAYS:

  ✅  PCA successfully compressed 30 features into just a few
      principal components while preserving 95%+ of the information.

  ✅  The 2D PCA scatter plot showed clear visual separation between
      Malignant and Benign classes — confirming meaningful structure.

  ✅  Most models maintained or IMPROVED accuracy after PCA because:
      - Noise/redundant features were removed
      - Correlated features were de-correlated
      - Overfitting risk was reduced (fewer dimensions)

  ✅  Training was faster with fewer input dimensions.

  ⚠️  IMPORTANT: PCA components are NOT interpretable as original
      features — there is a tradeoff between compression and
      feature interpretability.

  ⚠️  Always fit PCA on TRAINING data only (no data leakage).
      Then transform both train and test sets.

  📌  WHEN TO USE PCA:
      • High-dimensional datasets (100+ features)
      • Features are highly correlated
      • Need fast training (real-time systems)
      • Want to visualize high-dim data in 2D/3D
      • Model is overfitting due to too many features
""")
print('█' * 65)

In [ ]:
# ============================================================
# SECTION 13B: FINAL COMPREHENSIVE COMPARISON CHART
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ---- 1. Accuracy Comparison ----
x = np.arange(4)
w = 0.35
axes[0, 0].bar(x - w/2, acc_before, w, color='#3498db', label='Before PCA', alpha=0.9, edgecolor='navy')
axes[0, 0].bar(x + w/2, acc_after, w, color='#e74c3c', label='After PCA', alpha=0.9, edgecolor='darkred')
axes[0, 0].set_xticks(x); axes[0, 0].set_xticklabels(short_names)
axes[0, 0].set_ylabel('Accuracy (%)'); axes[0, 0].set_title('Test Accuracy Comparison', fontweight='bold')
axes[0, 0].legend(); axes[0, 0].set_ylim([85, 102])

# ---- 2. Accuracy vs Components ----
axes[0, 1].plot(list(component_range), lr_accuracies, 'b-', linewidth=2, label='LR')
axes[0, 1].plot(list(component_range), svm_accuracies, 'r-', linewidth=2, label='SVM')
axes[0, 1].axvline(x=n_95, color='green', linestyle='--', label=f'{n_95} comp (95% var)')
axes[0, 1].set_xlabel('PCA Components'); axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Accuracy vs PCA Components', fontweight='bold')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# ---- 3. Scree Plot ----
axes[1, 0].bar(range(1, 11), explained_var[:10] * 100, color='steelblue', edgecolor='navy', alpha=0.8)
axes[1, 0].set_xlabel('Component'); axes[1, 0].set_ylabel('Explained Variance (%)')
axes[1, 0].set_title('Scree Plot (Top 10 PCs)', fontweight='bold')

# ---- 4. Cumulative Variance ----
axes[1, 1].plot(range(1, 31), cumulative_var * 100, 'b-o', markersize=4)
axes[1, 1].axhline(y=95, color='red', linestyle='--', label='95% threshold')
axes[1, 1].axvline(x=n_95, color='green', linestyle='--', label=f'{n_95} components')
axes[1, 1].fill_between(range(1, n_95+1), cumulative_var[:n_95] * 100, alpha=0.2, color='red')
axes[1, 1].set_xlabel('Components'); axes[1, 1].set_ylabel('Cumulative Variance (%)')
axes[1, 1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('PCA Complete Analysis Dashboard — Breast Cancer Dataset',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🎓 Notebook complete! You have successfully:')
print('   1. Applied PCA to a 30-feature dataset')
print('   2. Visualized the 2D and 3D PCA projections')
print('   3. Trained 4 classifiers before and after PCA')
print('   4. Compared performance metrics comprehensively')
print('   5. Found the optimal number of PCA components')